In [9]:
import pandas as pd
from tensorflow.keras.models import load_model
import pickle

In [10]:
# Load the trained model, scaler pickle, onehot
model = load_model("model.h5")

# Load the encoder and scaler
with open("onehot_encoder_geo.pkl", "rb") as file:
    ohe_geo = pickle.load(file)

with open("label_encoder_gender.pkl", "rb") as file:
    le_gender = pickle.load(file)

with open("scaler.pkl", "rb") as file:
    scaler = pickle.load(file)

In [11]:
# Example input data
input_data = {
    "CreditScore": 600,
    "Geography": "France",
    "Gender": "Male",
    "Age": 40,
    "Tenure": 3,
    "Balance": 60000,
    "NumOfProducts": 2,
    "HasCrCard": 1,
    "IsActiveMember": 1,
    "EstimatedSalary": 50000
}

In [12]:
# OneHot Encode "Geography"
geo_encoded = ohe_geo.transform([[input_data["Geography"]]])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=ohe_geo.get_feature_names_out(["Geography"])
)
geo_encoded_df

c:\Users\91743\Documents\ANN Classification\ann_venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [13]:
# Changing the format to the dataframe
input_df = pd.DataFrame([input_data])

# Encoding the categorical variables
input_df["Gender"] = le_gender.transform(input_df["Gender"])

# Combining one-hot encoded columns with input data
input_df = pd.concat(
    [input_df.drop("Geography", axis=1), geo_encoded_df], axis=1
)

# Print
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [14]:
# Scaling the input data
input_scaled = scaler.transform(input_df)
print(input_scaled)

[[-0.53598516  0.91324755  0.10479359 -0.69539349 -0.25781119  0.80843615
   0.64920267  0.97481699 -0.87683221  1.00150113 -0.57946723 -0.57638802]]


In [15]:
# Predict Churn
prediction = model.predict(input_scaled)
print(prediction)

1/1 [==============================] - 0s 171ms/step
[[0.0304025]]


In [16]:
prediction_probability = prediction[0][0]

# Decision Making
if prediction_probability > 0.5:
    print("The customer is likely to churn")
else:
    print("The customer isn't likely to churn")

The customer isn't likely to churn
